# 数据检查与共享五折验证

负责人：A主实现，B/C复核。这是中文教学参考，正式实现由成员理解后编写、执行并核对。

输入挂载：官方比赛数据，以及历史收入邻域模型和混合特征模型的已保存输出。无需挂载旧track代码包。CPU训练，四线程；机器等待另计。

从已有OOF恢复历史分组，避免重新随机划分造成不可比较。历史预测仅用于提取分组和后续核对，新的模型会重新训练。


- [比赛数据与规则](https://www.kaggle.com/competitions/playground-series-s6e9)
- [LightGBM论文](https://proceedings.neurips.cc/paper/2017/hash/6449f44a102fde848669bdd9eb6b76fa-Abstract.html)
- [目标编码：内部交叉拟合与平滑](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.TargetEncoder.html)
- [AUC定义](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.roc_auc_score.html)

本教程生成时尚未执行Kaggle完整训练。历史分数是核对参照，不是本轮结果。阅读当前文档不意味着升级历史环境。

## 如何学习本文件

每次只运行一个单元，先用自己的话预测输出。`iloc`按位置取行，`loc`按标签取行；`to_numpy`去掉索引，之后必须保证位置对应。`assert`是验收条件，失败应查数据而非删除检查。`fit`从数据学习，`transform`使用已学习规则。

编程练习：修改一个小例子的输入并解释变化；正式配置保持历史定义。复杂特征组的整体增益不能归因于单一列。

## 读取数据与定义对齐函数

把行编号作为连接键；标签映射必须完整。输出应显示读取成功，没有目标缺失。

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from time import perf_counter
import json
import gc
import numpy as np
import pandas as pd
import sklearn
import lightgbm as lgb
from IPython.display import display
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import TargetEncoder
from sklearn.model_selection import StratifiedKFold

INPUT = Path('/kaggle/input')
TARGET = 'Will_Buy_EV'
SEED = 42
N_SPLITS = 5

def unique_file(name):
    """从已挂载输入中定位唯一文件；多个版本时停止，防止静默读错。"""
    paths = list(INPUT.rglob(name))
    assert len(paths) == 1, f'Expected one {name}, found {paths}'
    return paths[0]

competition_dirs = [INPUT/'competitions/playground-series-s6e9', INPUT/'playground-series-s6e9']
available = [p for p in competition_dirs if (p/'train.csv').is_file()]
assert len(available) == 1, 'Attach the official competition data.'
DATA = available[0]
train = pd.read_csv(DATA/'train.csv')
test = pd.read_csv(DATA/'test.csv')
sample = pd.read_csv(DATA/'sample_submission.csv')
y = train[TARGET].map({'No':0, 'Yes':1})
assert y.notna().all() and set(y.unique()) == {0,1}
assert train.id.is_unique and test.id.is_unique
assert sample.columns.tolist() == ['id',TARGET] and sample.id.equals(test.id)
assert train.columns.drop(['id',TARGET]).tolist() == test.columns.drop('id').tolist()

def align_rows(frame, ids):
    """先检查一一对应，再按官方顺序排列；不能直接假设CSV行序相同。"""
    assert frame.id.is_unique and len(frame) == len(ids)
    assert set(frame.id) == set(ids)
    return frame.set_index('id').loc[ids].reset_index()

def current_versions():
    return {'lightgbm':lgb.__version__, 'sklearn':sklearn.__version__,
            'numpy':np.__version__, 'pandas':pd.__version__}

## 明确历史文件路径

下面路径来自已运行的最终方案。若Kaggle挂载路径不同，只修改这两行，指向包含OOF、submission和run_summary的目录。必须先保存原Notebook输出并Add Input；上传ipynb不会附带CSV。

In [ ]:
history_neighborhood = INPUT/'notebooks/nemonade/10-multiscale-encoding/R003_multiscale_encoding_20260909_071033_059889'
history_hybrid = INPUT/'notebooks/nemonade/22-hyl001-hybrid-lightgbm/HYL001_hybrid_lightgbm_20260911_153854_073213'
history = {}
for name, directory in [('neighborhood',history_neighborhood),('hybrid',history_hybrid)]:
    assert directory.is_dir(), f'Attach saved output: {directory}'
    oof = align_rows(pd.read_csv(directory/'oof_predictions.csv'),train.id)
    assert np.array_equal(oof.target,y)
    assert oof.fold.isin(range(5)).all() and set(oof.fold)==set(range(5))
    pred_col = 'probability' if 'probability' in oof else 'prediction'
    assert oof[pred_col].between(0,1).all()
    summary = json.loads((directory/'run_summary.json').read_text())
    history[name] = {'directory':str(directory),'oof':oof,'summary':summary}
    print(name, roc_auc_score(y,oof[pred_col]))
assert np.array_equal(history['neighborhood']['oof'].fold,history['hybrid']['oof'].fold)
shared = history['neighborhood']['oof'][['id','target','fold']].copy()
display(shared.groupby('fold').agg(rows=('id','size'),positive_rate=('target','mean')))

## 冻结共享文件

输出只包含分组和数据说明。没有模型预测、代码哈希或提交名单。环境版本来自两次历史运行，后续训练前逐项核对。

In [ ]:
output = Path('/kaggle/working/team_foundation')
output.mkdir(exist_ok=False)
shared.to_csv(output/'shared_folds.csv',index=False)
note = {'target_mapping':{'No':0,'Yes':1}, 'train_rows':len(train),'test_rows':len(test),
        'feature_columns':test.columns.drop('id').tolist(),
        'fold_source':history['neighborhood']['directory'],
        'historical_versions':{k:v['summary'].get('versions',{}) for k,v in history.items()},
        'historical_features':{k:v['summary'].get('fold_features',{}) for k,v in history.items()},
        'historical_params':{k:v['summary'].get('model_params',{}) for k,v in history.items()},
        'history_directories':{k:v['directory'] for k,v in history.items()},
        'versions_when_preparing':current_versions()}
(output/'dataset_note.json').write_text(json.dumps(note,indent=2),encoding='utf-8')
print(output)

## 交接与理解检查

将`team_foundation`中的两个文件保存为一个固定版本Kaggle Dataset，三人挂载同一版本。已存在输出目录时先保存当前运行，再重开会话，避免覆盖。

B检查每行标签和fold；C故意打乱共享文件行序后用`align_rows`恢复并验证一致。问题：相同seed为什么不保证不同原始行序下分组相同？历史OOF为什么不能当成本轮新训练结果？